# 🛰️ Object Detection with YOLO — Detecting Buildings, Vehicles & Trees from Above
### GeoAI Professional Program · OrbitalIQ Analytics

**Today's goal:** train a YOLO model that looks at an aerial tile and draws a box around every **building**, **vehicle** and **tree** — then turn those detections into numbers a planner can use.

| Task | Question it answers | Output | Model we used |
|---|---|---|---|
| Classification | *What is in this image?* | One label per image | Random Forest, KNN |
| Segmentation | *Which pixels are building?* | A mask | **U-Net** (last class) |
| **Object detection** | ***Where is each object, and how many?*** | **Boxes + class + confidence** | **YOLO (today)** |

**YOLO = "You Only Look Once"** — the network sees the whole image a single time and predicts all boxes at once, which is why it is fast enough for real-time and large-area mapping.

> 💻 **This version runs in JupyterLab on a normal laptop CPU — no GPU needed.**
> Training is set to a light configuration (10 epochs) so it finishes in roughly 10–25 minutes depending on your laptop.

## Step 1 — Install PyTorch (CPU) and Ultralytics

Run this **once**. We install the **CPU-only** build of PyTorch first — it is about 200 MB instead of 2+ GB for the GPU build.
After it finishes: **Kernel → Restart Kernel**, then continue from the next cell.

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install ultralytics opencv-python matplotlib pandas

In [ ]:
import os, torch, ultralytics
print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version   :", torch.__version__)
print("CPU cores         :", os.cpu_count())
print("Running on        : CPU")

## ⚙️ Training settings for CPU

Change these numbers to fit your laptop. Everything later in the notebook uses them.

| Setting | Value | Why |
|---|---|---|
| `DEVICE` | `"cpu"` | No GPU needed |
| `EPOCHS` | `10` | Enough for good results on this dataset; 3 = quick demo, 20 = better |
| `IMGSZ` | `640` | Use `416` if your laptop is slow (vehicles get harder to see) |
| `BATCH` | `8` | Lower to `4` if you run out of memory |
| `WORKERS` | `0` | Avoids data-loader freezes in Jupyter on Windows |

In [ ]:
DEVICE  = "cpu"
EPOCHS  = 10
IMGSZ   = 640
BATCH   = 8
WORKERS = 0

torch.set_num_threads(os.cpu_count())   # use all CPU cores

## Step 2 — Get the dataset

We use a small **satellite-style demo dataset** (640 × 640 px tiles) with three classes:

| ID | Class | What it looks like from above |
|---|---|---|
| 0 | building | Rectangular roofs, different colours, a shadow on one side |
| 1 | vehicle | Small rectangles sitting on roads |
| 2 | tree | Round green canopies with a shadow |

**Option A (recommended in class):** run the cell below — it generates the dataset in ~20 seconds.
**Option B:** put `aerial_yolo_dataset.zip` in the same folder as this notebook and run the *Option B* cell instead.

In [ ]:
# ===== OPTION A: generate the dataset with code =====
"""
OrbitalIQ GeoAI Program - YOLO Demo Dataset Generator
-----------------------------------------------------
Creates a small synthetic "satellite-style" dataset in YOLO format.

Classes:
    0 = building   (roofs of different colours, with shadows)
    1 = vehicle    (small cars parked on roads)
    2 = tree       (round green canopies with shadows)

Output layout (exactly what YOLO expects):
    aerial_yolo_dataset/
        images/train  images/val  images/test
        labels/train  labels/val  labels/test
        data.yaml

Each label file has one line per object:
    class_id  x_center  y_center  width  height     (all normalised 0-1)
"""
import os
import random
import numpy as np
import cv2

CLASSES = ["building", "vehicle", "tree"]
IMG_SIZE = 640

ROOF_COLOURS = [  # BGR
    (60, 60, 170), (70, 90, 190), (200, 200, 205), (175, 180, 185),
    (140, 120, 90), (90, 110, 120), (210, 190, 150), (80, 80, 80),
]
CAR_COLOURS = [(240, 240, 240), (30, 30, 30), (40, 40, 200), (180, 90, 30),
               (150, 150, 150), (30, 150, 220)]


def make_background(rng):
    """Soil / grass texture made from smoothed random noise."""
    base = np.array(random.choice([(70, 110, 95), (80, 120, 120), (65, 100, 80),
                                   (95, 130, 140)]), np.float32)
    img = np.ones((IMG_SIZE, IMG_SIZE, 3), np.float32) * base
    for k, amp in [(81, 16), (21, 10), (3, 8)]:
        noise = rng.normal(0, 1, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
        noise = cv2.GaussianBlur(noise, (k, k), 0)
        noise = noise / (noise.std() + 1e-6) * amp
        img += noise[..., None]
    return img


def draw_roads(img, rng):
    """Draws 1-3 straight roads and returns their centre lines."""
    roads = []
    for _ in range(rng.integers(1, 4)):
        w = int(rng.integers(26, 40))
        if rng.random() < 0.5:   # horizontal
            y = int(rng.integers(60, IMG_SIZE - 60))
            cv2.rectangle(img, (0, y - w // 2), (IMG_SIZE, y + w // 2), (105, 105, 108), -1)
            cv2.line(img, (0, y), (IMG_SIZE, y), (190, 190, 190), 1)
            roads.append(("h", y, w))
        else:                    # vertical
            x = int(rng.integers(60, IMG_SIZE - 60))
            cv2.rectangle(img, (x - w // 2, 0), (x + w // 2, IMG_SIZE), (105, 105, 108), -1)
            cv2.line(img, (x, 0), (x, IMG_SIZE), (190, 190, 190), 1)
            roads.append(("v", x, w))
    return roads


def overlaps(box, boxes, pad=4):
    x1, y1, x2, y2 = box
    for a1, b1, a2, b2 in boxes:
        if x1 < a2 + pad and x2 + pad > a1 and y1 < b2 + pad and y2 + pad > b1:
            return True
    return False


def on_road(box, roads):
    x1, y1, x2, y2 = box
    for d, c, w in roads:
        if d == "h" and y1 < c + w // 2 + 3 and y2 > c - w // 2 - 3:
            return True
        if d == "v" and x1 < c + w // 2 + 3 and x2 > c - w // 2 - 3:
            return True
    return False


def add_building(img, rng, occupied, roads):
    for _ in range(40):
        w, h = int(rng.integers(30, 95)), int(rng.integers(30, 95))
        x1, y1 = int(rng.integers(5, IMG_SIZE - w - 5)), int(rng.integers(5, IMG_SIZE - h - 5))
        box = (x1, y1, x1 + w, y1 + h)
        if overlaps(box, occupied, 8) or on_road(box, roads):
            continue
        sh = int(rng.integers(4, 9))  # shadow towards bottom-right
        cv2.rectangle(img, (x1 + sh, y1 + sh), (x1 + w + sh, y1 + h + sh), (35, 45, 40), -1)
        col = np.array(random.choice(ROOF_COLOURS), np.float32)
        cv2.rectangle(img, (x1, y1), (x1 + w, y1 + h), col.tolist(), -1)
        # gable roof: one half slightly darker
        if rng.random() < 0.6:
            if w > h:
                cv2.rectangle(img, (x1, y1 + h // 2), (x1 + w, y1 + h), (col * 0.82).tolist(), -1)
            else:
                cv2.rectangle(img, (x1 + w // 2, y1), (x1 + w, y1 + h), (col * 0.82).tolist(), -1)
        cv2.rectangle(img, (x1, y1), (x1 + w, y1 + h), (col * 0.6).tolist(), 1)
        occupied.append(box)
        return box
    return None


def add_vehicle(img, rng, occupied, roads):
    if not roads:
        return None
    for _ in range(40):
        d, c, w = roads[rng.integers(len(roads))]
        L, W = int(rng.integers(14, 20)), int(rng.integers(7, 10))
        lane = c + (w // 4 if rng.random() < 0.5 else -w // 4)
        if d == "h":
            x1 = int(rng.integers(5, IMG_SIZE - L - 5))
            box = (x1, lane - W // 2, x1 + L, lane + W // 2)
        else:
            y1 = int(rng.integers(5, IMG_SIZE - L - 5))
            box = (lane - W // 2, y1, lane + W // 2, y1 + L)
        if overlaps(box, occupied, 3):
            continue
        x1, y1, x2, y2 = box
        col = random.choice(CAR_COLOURS)
        cv2.rectangle(img, (x1 + 2, y1 + 2), (x2 + 2, y2 + 2), (50, 50, 50), -1)
        cv2.rectangle(img, (x1, y1), (x2, y2), col, -1)
        # windscreen
        if d == "h":
            cv2.rectangle(img, (x1 + L // 3, y1 + 1), (x1 + L // 3 + 3, y2 - 1), (60, 50, 40), -1)
        else:
            cv2.rectangle(img, (x1 + 1, y1 + L // 3), (x2 - 1, y1 + L // 3 + 3), (60, 50, 40), -1)
        occupied.append(box)
        return box
    return None


def add_tree(img, rng, occupied, roads):
    for _ in range(40):
        r = int(rng.integers(9, 22))
        cx, cy = int(rng.integers(r + 5, IMG_SIZE - r - 5)), int(rng.integers(r + 5, IMG_SIZE - r - 5))
        box = (cx - r, cy - r, cx + r, cy + r)
        if overlaps(box, occupied, 2) or on_road(box, roads):
            continue
        cv2.circle(img, (cx + r // 3, cy + r // 3), r, (25, 40, 30), -1)          # shadow
        g = int(rng.integers(90, 140))
        cv2.circle(img, (cx, cy), r, (30, g, 40), -1)                              # canopy
        cv2.circle(img, (cx - r // 4, cy - r // 4), max(2, r // 2), (45, g + 40, 60), -1)  # sunlit
        occupied.append(box)
        return box
    return None


def make_image(seed):
    rng = np.random.default_rng(seed)
    random.seed(seed)
    img = make_background(rng)
    roads = draw_roads(img, rng)
    occupied, labels = [], []
    makers = [(0, add_building, rng.integers(4, 12)),
              (1, add_vehicle, rng.integers(2, 9)),
              (2, add_tree, rng.integers(4, 15))]
    for cls, fn, n in makers:
        for _ in range(int(n)):
            box = fn(img, rng, occupied, roads)
            if box:
                labels.append((cls, box))
    # mild sensor blur + noise so it looks less "drawn"
    img = cv2.GaussianBlur(img, (3, 3), 0)
    img += rng.normal(0, 4, img.shape).astype(np.float32)
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img, labels


def to_yolo(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2 / IMG_SIZE, (y1 + y2) / 2 / IMG_SIZE,
            (x2 - x1) / IMG_SIZE, (y2 - y1) / IMG_SIZE)


def build_dataset(root="aerial_yolo_dataset", n_train=200, n_val=50, n_test=10):
    splits = {"train": n_train, "val": n_val, "test": n_test}
    seed = 0
    for split, n in splits.items():
        os.makedirs(f"{root}/images/{split}", exist_ok=True)
        os.makedirs(f"{root}/labels/{split}", exist_ok=True)
        for i in range(n):
            img, labels = make_image(seed)
            seed += 1
            name = f"tile_{split}_{i:04d}"
            cv2.imwrite(f"{root}/images/{split}/{name}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 92])
            with open(f"{root}/labels/{split}/{name}.txt", "w") as f:
                for cls, box in labels:
                    xc, yc, w, h = to_yolo(box)
                    f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
    with open(f"{root}/data.yaml", "w") as f:
        abs_root = os.path.abspath(root).replace("\\", "/")   # forward slashes work on Windows too
        f.write(f"path: '{abs_root}'\n")
        f.write("train: images/train\nval: images/val\ntest: images/test\n\n")
        f.write(f"nc: {len(CLASSES)}\n")
        f.write("names:\n")
        for i, c in enumerate(CLASSES):
            f.write(f"  {i}: {c}\n")
    print(f"Dataset ready in '{root}':", splits)


build_dataset()   # 200 train / 50 val / 10 test tiles

In [ ]:
# ===== OPTION B: use the zip file instead (skip if you ran Option A) =====
# import zipfile, os, re
# zipfile.ZipFile("aerial_yolo_dataset.zip").extractall(".")
# # the yaml must point at the folder's real location on YOUR computer
# root = os.path.abspath("aerial_yolo_dataset").replace("\\", "/")
# y = open("aerial_yolo_dataset/data.yaml").read()
# y = re.sub(r"^path:.*$", f"path: '{root}'", y, flags=re.M)
# open("aerial_yolo_dataset/data.yaml", "w").write(y)
# print(y)

## Step 3 — Understand the YOLO folder structure

YOLO is strict about layout: **every image has a text file with the same name** in the matching `labels` folder.

```
aerial_yolo_dataset/
├── images/  train/  val/  test/      ← .jpg tiles
├── labels/  train/  val/  test/      ← .txt files, one per image
└── data.yaml                         ← tells YOLO where things are + class names
```

In [ ]:
import os, glob
DATA = "aerial_yolo_dataset"
for split in ["train", "val", "test"]:
    n_img = len(glob.glob(f"{DATA}/images/{split}/*.jpg"))
    n_lbl = len(glob.glob(f"{DATA}/labels/{split}/*.txt"))
    print(f"{split:5s}: {n_img} images | {n_lbl} label files")

print("\n----- data.yaml -----")
print(open(f"{DATA}/data.yaml").read())

## Step 4 — Read one label file (the most important concept today)

Each line = **one object**:

```
class_id   x_center   y_center   width   height
```

All four numbers are **normalised between 0 and 1** (divided by image width/height), so the same label works at any image size.

*Example:* `0 0.25 0.40 0.10 0.08` → a **building** whose centre is 25 % across and 40 % down, and whose box is 10 % of the image wide and 8 % tall.

In [ ]:
sample = sorted(glob.glob(f"{DATA}/labels/train/*.txt"))[0]
print("File:", sample, "\n")
lines = open(sample).read().strip().split("\n")
for line in lines[:8]:
    print(line)
print(f"... {len(lines)} objects in this tile")

# Convert the first line back to pixels
c, xc, yc, w, h = map(float, lines[0].split())
W = H = 640
print(f"\nFirst object → class {int(c)}, pixel box: "
      f"x1={int((xc-w/2)*W)}, y1={int((yc-h/2)*H)}, x2={int((xc+w/2)*W)}, y2={int((yc+h/2)*H)}")

## Step 5 — Visualise the ground truth

In [ ]:
import cv2, matplotlib.pyplot as plt

CLASS_NAMES = ["building", "vehicle", "tree"]
COLOURS = [(255, 60, 60), (0, 220, 255), (255, 230, 0)]   # RGB

def show_ground_truth(img_path, ax):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    lbl = img_path.replace("images", "labels").replace(".jpg", ".txt")
    for line in open(lbl):
        c, xc, yc, w, h = map(float, line.split())
        c = int(c)
        x1, y1 = int((xc - w/2) * W), int((yc - h/2) * H)
        x2, y2 = int((xc + w/2) * W), int((yc + h/2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLOURS[c], 2)
    ax.imshow(img); ax.axis("off"); ax.set_title(os.path.basename(img_path), fontsize=9)

paths = sorted(glob.glob(f"{DATA}/images/train/*.jpg"))[:4]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for p, ax in zip(paths, axes):
    show_ground_truth(p, ax)
plt.suptitle("Ground truth  —  red: building | cyan: vehicle | yellow: tree", fontsize=13)
plt.tight_layout(); plt.show()

## Step 6 — How balanced is the dataset?

In [ ]:
import collections, pandas as pd

counts = collections.Counter()
sizes = collections.defaultdict(list)
for f in glob.glob(f"{DATA}/labels/train/*.txt"):
    for line in open(f):
        c, _, _, w, h = map(float, line.split())
        counts[CLASS_NAMES[int(c)]] += 1
        sizes[CLASS_NAMES[int(c)]].append(w * h * 640 * 640)

df = pd.DataFrame({"objects": counts,
                   "avg_area_px": {k: round(sum(v)/len(v)) for k, v in sizes.items()}})
print(df)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["objects"].plot.bar(ax=ax[0], color=["#d9534f", "#17a2b8", "#e0b000"], title="Objects per class (train)")
df["avg_area_px"].plot.bar(ax=ax[1], color=["#d9534f", "#17a2b8", "#e0b000"], title="Average box area (pixels²)")
for a in ax: a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

💡 **Notice:** vehicles are **tiny** (~120 px²) compared with buildings (~3,700 px²). Small objects are the hardest thing in remote-sensing detection — watch the vehicle score later.

## Step 7 — Load a pre-trained YOLO model (transfer learning)

`yolo11n.pt` = YOLO11 **nano**, pre-trained on the COCO dataset (80 everyday classes).
We **fine-tune** it on our aerial classes instead of training from zero — just as we reuse knowledge in real projects.

| Model | Size | Speed | Accuracy |
|---|---|---|---|
| yolo11**n** | 2.6 M params | fastest | good |
| yolo11**s** | 9.4 M | fast | better |
| yolo11**m** | 20 M | medium | high |

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.info()

## Step 8 — Train 🚀

| Parameter | Meaning |
|---|---|
| `data` | path to `data.yaml` |
| `epochs` | how many times the model sees the full training set |
| `imgsz` | tiles are resized to this size |
| `batch` | images processed together |
| `patience` | stop early if validation stops improving |

| `device` | `"cpu"` — train on the processor |
| `workers` | `0` — load data in the main process (safe in Jupyter) |

⏱️ On a laptop CPU, each epoch takes about **1–2 minutes** with 200 tiles, so 10 epochs ≈ 10–25 minutes.
You can watch the progress bar — `box_loss` and `cls_loss` should fall every epoch.

ℹ️ If you see `WARNING NMS time limit exceeded` — that is normal on CPU and does not affect training.

In [ ]:
results = model.train(
    data=f"{DATA}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=5,
    project="runs",
    name="aerial_yolo",
    exist_ok=True,
    plots=True,
)
RUN = str(model.trainer.save_dir)      # folder where weights, plots and logs were saved
print("Results saved in:", RUN)

## Step 9 — Read the training curves

* **box_loss** – how wrong the box positions are ↓
* **cls_loss** – how wrong the class guesses are ↓
* **mAP50** – accuracy when a box counts as correct if it overlaps the truth by ≥ 50 % ↑
* **mAP50-95** – stricter average over overlaps 50 % … 95 % ↑

Train loss falling **and** validation loss falling = the model is learning, not memorising.

In [ ]:
from IPython.display import Image, display
display(Image(f"{RUN}/results.png", width=1000))

## Step 10 — Evaluate on the validation set

**IoU (Intersection over Union)** = overlap area ÷ union area of predicted and true box.

| Metric | Plain-English meaning |
|---|---|
| **Precision** | Of the boxes the model drew, how many were real objects? |
| **Recall** | Of the real objects, how many did the model find? |
| **mAP50** | Overall detection quality at IoU ≥ 0.5 (the headline number) |

In [ ]:
best = YOLO(f"{RUN}/weights/best.pt")
metrics = best.val(data=f"{DATA}/data.yaml", split="val", imgsz=IMGSZ, batch=BATCH,
                   device=DEVICE, workers=WORKERS, plots=True)

rows = []
for i, name in enumerate(CLASS_NAMES):
    p, r, ap50, ap = metrics.box.class_result(i)
    rows.append([name, round(p, 3), round(r, 3), round(ap50, 3), round(ap, 3)])
table = pd.DataFrame(rows, columns=["class", "precision", "recall", "mAP50", "mAP50-95"])
print(table.to_string(index=False))
print(f"\nOverall  mAP50 = {metrics.box.map50:.3f}   |   mAP50-95 = {metrics.box.map:.3f}")

In [ ]:
display(Image(f"{metrics.save_dir}/confusion_matrix_normalized.png", width=600))

## Step 11 — Detect objects on unseen test tiles

In [ ]:
test_imgs = sorted(glob.glob(f"{DATA}/images/test/*.jpg"))
preds = best.predict(test_imgs, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for r, ax in zip(preds[:6], axes.ravel()):
    ax.imshow(r.plot(line_width=2, font_size=10)[..., ::-1])   # BGR → RGB
    ax.set_title(f"{os.path.basename(r.path)} — {len(r.boxes)} detections")
    ax.axis("off")
plt.tight_layout(); plt.show()

## Step 12 — Ground truth vs prediction, side by side

In [ ]:
r = preds[0]
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
show_ground_truth(r.path, ax[0]); ax[0].set_title("Ground truth (human labels)")
ax[1].imshow(r.plot(line_width=2)[..., ::-1]); ax[1].axis("off"); ax[1].set_title("YOLO prediction")
plt.tight_layout(); plt.show()

## Step 13 — From boxes to GeoAI insight: counting per tile

A planner doesn't want boxes — they want **numbers**: how many buildings, how many vehicles, how much tree cover in each tile.

In [ ]:
records = []
for r in preds:
    cls = r.boxes.cls.cpu().numpy().astype(int)
    xywh = r.boxes.xywh.cpu().numpy()
    areas = xywh[:, 2] * xywh[:, 3]
    tile_area = r.orig_shape[0] * r.orig_shape[1]
    records.append({
        "tile": os.path.basename(r.path),
        "buildings": int((cls == 0).sum()),
        "vehicles": int((cls == 1).sum()),
        "trees": int((cls == 2).sum()),
        "built_up_%": round(100 * areas[cls == 0].sum() / tile_area, 1),
        "canopy_%": round(100 * areas[cls == 2].sum() / tile_area, 1),
    })
summary = pd.DataFrame(records)
summary

In [ ]:
summary.set_index("tile")[["buildings", "vehicles", "trees"]].plot.bar(
    figsize=(14, 5), color=["#d9534f", "#17a2b8", "#e0b000"], title="Detected objects per test tile")
plt.xticks(rotation=45, ha="right"); plt.ylabel("count"); plt.tight_layout(); plt.show()

summary.to_csv("tile_object_counts.csv", index=False)
print("Saved tile_object_counts.csv")

## Step 14 — Experiment: what does the confidence threshold do?

`conf` = the minimum score a box needs to be kept.
* **Low conf** → more boxes, more false alarms (higher recall, lower precision)
* **High conf** → fewer boxes, may miss objects (higher precision, lower recall)

In [ ]:
img = test_imgs[1]
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for conf, ax in zip([0.05, 0.25, 0.70], axes):
    r = best.predict(img, conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    ax.imshow(r.plot(line_width=2, labels=False)[..., ::-1])
    ax.set_title(f"conf = {conf}  →  {len(r.boxes)} boxes", fontsize=14); ax.axis("off")
plt.tight_layout(); plt.show()

## Step 15 — Save your model
`best.pt` is your trained detector — you can load it later with `YOLO("best.pt")` on any new tile.

In [ ]:
import shutil
shutil.copy(f"{RUN}/weights/best.pt", "aerial_yolo_best.pt")
print("Model saved as:", os.path.abspath("aerial_yolo_best.pt"))
print("Counts saved as:", os.path.abspath("tile_object_counts.csv"))

## 🔑 Key takeaways
1. **Detection = where + what + how many**; segmentation (U-Net) = which pixels.
2. YOLO labels are `class xc yc w h`, normalised 0–1, one `.txt` per image.
3. `data.yaml` connects images, labels and class names.
4. **Transfer learning** — start from pre-trained weights, fine-tune on our classes.
5. Judge the model with **precision, recall, mAP50**; small objects (vehicles) are hardest.
6. Real value comes after detection: **counts and densities per tile** for planning.

## 📝 Practice
1. Change `EPOCHS` from 10 to **3** and then **20** — how much does mAP50 change? Is the extra CPU time worth it?
2. Swap `yolo11n.pt` for `yolo11s.pt` — compare accuracy and **training time per epoch** on CPU.
3. Train with `imgsz=320` — what happens to the **vehicle** score, and why?
4. Add a filter: which test tile has the **highest built-up %**?
5. **Real-world step:** export building footprints you digitised in QGIS as YOLO boxes and train on a Cartosat-3 tile.